In [3]:
!pip install dash dash-bootstrap-components plotly pandas numpy -q

In [6]:
app.run(jupyter_mode="external", debug=True, port=8050)

AssertionError: The setup method 'errorhandler' can no longer be called on the application. It has already handled its first request, any changes will not be applied consistently.
Make sure all imports, decorators, functions, etc. needed to set up the application are done before running it.

In [5]:
# -*- coding: utf-8 -*-
"""
Dashboard interactivo estilo "Spotify Analytics" hecho con Dash + Plotly.

Cómo ejecutarlo:
    pip install dash plotly pandas numpy
    python spotify_dashboard.py

Luego abre en el navegador: http://127.0.0.1:8050

NOTA: Los datos usados aquí son de EJEMPLO (generados/aproximados a partir de
la imagen de referencia). Si tienes tu propio dataset (por ejemplo un CSV de
Spotify con columnas como track_name, artist, streams, released_year,
in_apple_playlists, mode, etc.), reemplaza la sección "DATOS" de más abajo
por la carga de tu archivo real, por ejemplo:

    df = pd.read_csv("spotify_data.csv")
"""

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc


# =========================================================
# 1. DATOS (reemplaza esto por tu dataset real si lo tienes)
# =========================================================

# --- Total de streams y artistas top ---
TOTAL_STREAMS_TXT = "490 billion"

top_artists = [
    {"name": "The Weeknd", "img": "https://i.scdn.co/image/ab6761610000e5eb8ea925fdcdd6f202fdaeb00c"},
    {"name": "Taylor Swift", "img": "https://i.scdn.co/image/ab6761610000e5eb859e4f24397a9d5325f4c5a4"},
    {"name": "Ed Sheeran", "img": "https://i.scdn.co/image/ab6761610000e5eb3bcef85e105dfc42399ef0ba"},
    {"name": "Harry Styles", "img": "https://i.scdn.co/image/ab6761610000e5eb472251d942910d38086ee0b3"},
    {"name": "Bad Bunny", "img": "https://i.scdn.co/image/ab6761610000e5eb4293385d324db8558179afd9"},
]

# --- Crecimiento general: streams acumulados por año de lanzamiento ---
np.random.seed(42)
years = np.arange(1930, 2024)
growth = np.concatenate([
    np.random.uniform(0, 0.5e9, size=len(years[years < 1990])),
    np.random.uniform(0.5e9, 5e9, size=len(years[(years >= 1990) & (years < 2010)])),
    np.linspace(5e9, 95e9, num=len(years[years >= 2010])) + np.random.uniform(-3e9, 3e9, size=len(years[years >= 2010]))
])
growth = np.clip(growth, 0, None)
df_growth = pd.DataFrame({"released_year": years, "streams": growth})

# --- Top tracks por año (varios años para el dropdown) ---
track_pool = {
    2019: [
        ("Blinding Lights", "The Weeknd", 3700),
        ("Dance Monkey", "Tones and I", 2900),
        ("Señorita", "Shawn Mendes, Camila Cabello", 2450),
        ("Watermelon Sugar", "Harry Styles", 2200),
        ("Don't Start Now", "Dua Lipa", 2150),
    ],
    2020: [
        ("Blinding Lights", "The Weeknd", 3400),
        ("Mood", "24kGoldn ft. iann dior", 2600),
        ("Watermelon Sugar", "Harry Styles", 2500),
        ("Dynamite", "BTS", 2100),
        ("Positions", "Ariana Grande", 1900),
    ],
    2021: [
        ("Stay", "The Kid LAROI, Justin Bieber", 3100),
        ("good 4 u", "Olivia Rodrigo", 2800),
        ("Levitating", "Dua Lipa", 2600),
        ("Bad Habits", "Ed Sheeran", 2300),
        ("Save Your Tears", "The Weeknd", 2150),
    ],
    2022: [
        ("As It Was", "Harry Styles", 3600),
        ("Anti-Hero", "Taylor Swift", 2900),
        ("Titi Me Preguntó", "Bad Bunny", 2500),
        ("About Damn Time", "Lizzo", 2100),
        ("Unholy", "Sam Smith, Kim Petras", 2000),
    ],
}
years_dropdown = list(track_pool.keys())

# --- Categoría: modo Mayor / Menor ---
mode_counts = pd.DataFrame({
    "mode": ["Major", "Minor"],
    "count": [293620325410, 195835625001]  # valores de ejemplo (grandes, como IDs concatenados en la imagen)
})

# --- Top tracks por atributo (dropdown: playlists de apple, spotify, etc.) ---
attribute_pool = {
    "in_apple_playlists": [
        ("When I Was Your Man", 1550),
        ("Locked Out Of Heaven", 1400),
        ("Let Me Down Slowly", 1300),
        ("Talking To The Moon", 950),
        ("My Universe", 900),
    ],
    "in_spotify_playlists": [
        ("Shape of You", 1700),
        ("Blinding Lights", 1600),
        ("Dance Monkey", 1350),
        ("Someone You Loved", 1200),
        ("Perfect", 1100),
    ],
}
attribute_dropdown = list(attribute_pool.keys())

# --- Tracks poco populares ---
unpopular = pd.DataFrame({
    "track": ["Love Grows\n(Where My...)", "Que Vuelvas", "Jhoome Jo\nPathaan", "QUEMA", "Gol Bolinha,\nGol Quadr..."],
    "streams": [0.3e6, 0.4e6, 3e6, 9.5e6, 9.8e6],
    "note": ["", "", "", "", "1 null"]
})


# =========================================================
# 2. APP DASH
# =========================================================

app = Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])
app.title = "Spotify Analytics Dashboard"

GREEN_DARK = "#1b6e3c"
GREEN = "#1DB954"
GREEN_LIGHT = "#8fe3ab"
BG = "#f7f7f7"

card_style = {
    "backgroundColor": "white",
    "borderRadius": "16px",
    "padding": "20px",
    "boxShadow": "0 2px 8px rgba(0,0,0,0.06)",
    "height": "100%",
}


def artist_card(artist):
    return html.Div(
        [
            html.Img(
                src=artist["img"],
                style={
                    "width": "70px",
                    "height": "70px",
                    "borderRadius": "50%",
                    "objectFit": "cover",
                    "border": "3px solid " + GREEN,
                },
            ),
            html.P(artist["name"], style={"textAlign": "center", "marginTop": "8px", "fontWeight": "600"}),
        ],
        style={"textAlign": "center", "margin": "0 10px"},
    )


app.layout = dbc.Container(
    fluid=True,
    style={"backgroundColor": BG, "padding": "24px"},
    children=[
        # ---------- HEADER ----------
        dbc.Row(
            [
                dbc.Col(
                    html.Div(
                        [
                            html.Img(
                                src="https://upload.wikimedia.org/wikipedia/commons/2/26/Spotify_logo_with_text.svg",
                                style={"height": "40px"},
                            ),
                        ],
                        style={**card_style, "display": "flex", "alignItems": "center", "justifyContent": "center"},
                    ),
                    width=3,
                ),
                dbc.Col(
                    html.Div(
                        [
                            html.H5("Total Streams", style={"textAlign": "center", "fontWeight": "700"}),
                            html.Div(
                                TOTAL_STREAMS_TXT,
                                style={
                                    "backgroundColor": GREEN,
                                    "color": "white",
                                    "fontSize": "28px",
                                    "fontWeight": "800",
                                    "textAlign": "center",
                                    "borderRadius": "12px",
                                    "padding": "14px",
                                },
                            ),
                        ],
                        style=card_style,
                    ),
                    width=3,
                ),
                dbc.Col(
                    html.Div(
                        [
                            html.H5("Top Artists", style={"textAlign": "center", "fontWeight": "700"}),
                            html.Div(
                                [artist_card(a) for a in top_artists],
                                style={"display": "flex", "justifyContent": "space-around", "flexWrap": "wrap"},
                            ),
                        ],
                        style=card_style,
                    ),
                    width=6,
                ),
            ],
            className="mb-3 g-3",
        ),

        # ---------- FILA 2: Growth + Top tracks by year ----------
        dbc.Row(
            [
                dbc.Col(
                    html.Div(
                        [
                            html.H5("Overall Growth", style={"fontWeight": "700"}),
                            dcc.Graph(id="growth-chart", config={"displayModeBar": False}),
                        ],
                        style=card_style,
                    ),
                    width=6,
                ),
                dbc.Col(
                    html.Div(
                        [
                            dbc.Row(
                                [
                                    dbc.Col(html.H5("Top Tracks by Year", style={"fontWeight": "700"}), width=8),
                                    dbc.Col(
                                        dcc.Dropdown(
                                            id="year-dropdown",
                                            options=[{"label": str(y), "value": y} for y in years_dropdown],
                                            value=years_dropdown[0],
                                            clearable=False,
                                        ),
                                        width=4,
                                    ),
                                ]
                            ),
                            dcc.Graph(id="tracks-year-chart", config={"displayModeBar": False}),
                        ],
                        style=card_style,
                    ),
                    width=6,
                ),
            ],
            className="mb-3 g-3",
        ),

        # ---------- FILA 3: Category + Attribute + Unpopular ----------
        dbc.Row(
            [
                dbc.Col(
                    html.Div(
                        [
                            html.H5("Category", style={"fontWeight": "700"}),
                            dcc.Graph(id="mode-pie", config={"displayModeBar": False}),
                        ],
                        style=card_style,
                    ),
                    width=4,
                ),
                dbc.Col(
                    html.Div(
                        [
                            dbc.Row(
                                [
                                    dbc.Col(html.H5("Top Tracks by Attribute", style={"fontWeight": "700"}), width=8),
                                    dbc.Col(
                                        dcc.Dropdown(
                                            id="attribute-dropdown",
                                            options=[{"label": a, "value": a} for a in attribute_dropdown],
                                            value=attribute_dropdown[0],
                                            clearable=False,
                                        ),
                                        width=4,
                                    ),
                                ]
                            ),
                            dcc.Graph(id="attribute-chart", config={"displayModeBar": False}),
                        ],
                        style=card_style,
                    ),
                    width=4,
                ),
                dbc.Col(
                    html.Div(
                        [
                            html.H5("Unpopular tracks", style={"fontWeight": "700"}),
                            dcc.Graph(id="unpopular-chart", config={"displayModeBar": False}),
                        ],
                        style=card_style,
                    ),
                    width=4,
                ),
            ],
            className="g-3",
        ),
    ],
)


# =========================================================
# 3. CALLBACKS (interactividad)
# =========================================================

@app.callback(Output("growth-chart", "figure"), Input("growth-chart", "id"))
def update_growth(_):
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=df_growth["released_year"],
            y=df_growth["streams"],
            fill="tozeroy",
            mode="lines",
            line=dict(color=GREEN_DARK, width=1),
            fillcolor="rgba(46, 125, 50, 0.35)",
        )
    )
    fig.update_layout(
        margin=dict(l=10, r=10, t=10, b=10),
        xaxis_title="Released Year",
        yaxis_title="Streams",
        plot_bgcolor="white",
        height=300,
    )
    return fig


@app.callback(Output("tracks-year-chart", "figure"), Input("year-dropdown", "value"))
def update_tracks_by_year(year):
    data = track_pool[year]
    names = [f"{t[0]}" for t in data][::-1]
    streams = [t[2] for t in data][::-1]
    colors = [GREEN_DARK, GREEN, "#4fc97a", GREEN_LIGHT, "#c8f0d6"][::-1]
    fig = go.Figure(
        go.Bar(
            x=streams,
            y=names,
            orientation="h",
            marker_color=colors,
            text=[str(i) for i in range(len(names), 0, -1)],
            textposition="outside",
        )
    )
    fig.update_layout(
        margin=dict(l=10, r=30, t=10, b=10),
        xaxis_title="Streams",
        plot_bgcolor="white",
        height=300,
    )
    return fig


@app.callback(Output("mode-pie", "figure"), Input("mode-pie", "id"))
def update_pie(_):
    fig = px.pie(
        mode_counts,
        names="mode",
        values="count",
        color="mode",
        color_discrete_map={"Major": GREEN_DARK, "Minor": GREEN_LIGHT},
        hole=0,
    )
    fig.update_traces(textinfo="none")
    fig.update_layout(margin=dict(l=10, r=10, t=10, b=10), height=280)
    return fig


@app.callback(Output("attribute-chart", "figure"), Input("attribute-dropdown", "value"))
def update_attribute(attribute):
    data = attribute_pool[attribute]
    names = [t[0] for t in data]
    streams = [t[1] for t in data]
    colors = [GREEN_DARK, GREEN, "#4fc97a", GREEN_LIGHT, "#c8f0d6"]
    fig = go.Figure(go.Bar(x=names, y=streams, marker_color=colors))
    fig.update_layout(
        margin=dict(l=10, r=10, t=10, b=10),
        yaxis_title="Streams",
        plot_bgcolor="white",
        height=300,
        xaxis_tickangle=-15,
    )
    return fig


@app.callback(Output("unpopular-chart", "figure"), Input("unpopular-chart", "id"))
def update_unpopular(_):
    colors = [GREEN_LIGHT if v < 5e6 else GREEN_DARK if v < 6e6 else "#a8e6bd" for v in unpopular["streams"]]
    fig = go.Figure(
        go.Bar(
            x=unpopular["track"],
            y=unpopular["streams"],
            marker_color=colors,
            text=unpopular["note"],
            textposition="outside",
        )
    )
    fig.update_layout(
        margin=dict(l=10, r=10, t=10, b=10),
        yaxis_title="Streams",
        plot_bgcolor="white",
        height=300,
    )
    return fig


# =========================================================
# 4. RUN
# =========================================================

app.run(jupyter_mode="external", debug=True, port=8050)

# ---------------------------------------------------------
# NOTAS PARA GOOGLE COLAB:
# - Si prefieres verlo DENTRO de la celda en vez de un link, cambia la línea
#   de arriba por:
#     app.run(jupyter_mode="inline", debug=True, height=800)
# - Si el puerto 8050 ya está en uso, cambia port=8050 por otro, ej: port=8060
# ---------------------------------------------------------

Dash app running on:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [7]:
from google.colab import output
output.serve_kernel_port_as_iframe(8050, height=800)

<IPython.core.display.Javascript object>